In [ ]:
# ================================
# Improved Pairwise Comparison (AvalAI - Gemma) - Robust Error Handling
# Enhanced rate limiting, better error handling, and API compatibility
# ================================
import io
import json
import csv
import time
import random
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import requests
import os
import sys
import re
from datetime import datetime, timedelta

# Colab helpers (optional downloads)
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def load_api_key(name, fallback_names=()):
    candidates = (name, *fallback_names)
    repo_root = next(
        (
            candidate
            for candidate in (Path.cwd(), *Path.cwd().parents)
            if (candidate / "common" / "__init__.py").exists()
        ),
        None,
    )

    get_api_key = None
    if repo_root is not None:
        if str(repo_root) not in sys.path:
            sys.path.insert(0, str(repo_root))
        try:
            from common import get_api_key as shared_get_api_key
        except ImportError:
            pass
        else:
            get_api_key = shared_get_api_key

    if get_api_key is not None:
        for candidate in candidates:
            try:
                return get_api_key(candidate)
            except KeyError:
                pass

    for candidate in candidates:
        value = os.getenv(candidate, "").strip()
        if value:
            return value

    joined = ", ".join(candidates)
    raise RuntimeError(
        f"Missing API key. Set one of [{joined}] in the environment"
        " or the top-level config file."
    )

# ----------------------
# Enhanced User Configuration
# ----------------------
RANDOM_SEED = 7
JUDGE_MODE = "avalai"
OUTPUT_DIR = Path("/kaggle/working/pairwise_out")

ONE_SHOT_JSON_PATH = "/kaggle/input/rdf-evaluation/rdf_extractions_oneshot.json"
FEW_SHOT_JSON_PATH = "/kaggle/input/rdf-evaluation/rdf_extractions_fewshot.json"
IMAGES_DIR_PATH = "/kaggle/input/rdf-evaluation/dataset/dataset"

# Enhanced rate limiting and error handling
MAX_CHARS = 1500          # Further reduced to avoid 400 errors
MIN_SLEEP_BETWEEN_ITEMS = 2.0  # Minimum sleep between requests
MAX_SLEEP_BETWEEN_ITEMS = 5.0  # Maximum sleep between requests
PRINT_REQUEST_SIZES = True
PRINT_RESPONSE_SNIPPET = True
REQUEST_TIMEOUT = 120     # Increased timeout
MAX_RETRIES = 5           # Reduced retries but better logic
EXPONENTIAL_BACKOFF_BASE = 2.0
MAX_BACKOFF_DELAY = 300   # Maximum backoff delay (5 minutes)

# Rate limiting tracking
class RateLimiter:
    def __init__(self, min_interval=2.0, max_interval=5.0):
        self.min_interval = min_interval
        self.max_interval = max_interval
        self.last_request_time = 0
        self.consecutive_errors = 0
        self.success_count = 0

    def wait_if_needed(self):
        """Wait appropriate time before next request"""
        current_time = time.time()
        elapsed = current_time - self.last_request_time

        # Calculate required wait time based on error history
        if self.consecutive_errors > 0:
            # Increase wait time after errors
            required_wait = self.max_interval * (1.5 ** min(self.consecutive_errors, 5))
        else:
            # Normal wait time
            required_wait = random.uniform(self.min_interval, self.max_interval)

        if elapsed < required_wait:
            sleep_time = required_wait - elapsed
            print(f"⏳ Rate limiting: sleeping {sleep_time:.1f}s")
            time.sleep(sleep_time)

        self.last_request_time = time.time()

    def record_success(self):
        self.consecutive_errors = 0
        self.success_count += 1

    def record_error(self):
        self.consecutive_errors += 1

# ----------------------
# AvalAI settings
# ----------------------
AVALAI_API_KEY = load_api_key("AVALAI_API_KEY", ("OPENAI_API_KEY",))
BASE_URL = "https://api.avalai.ir/v1"
AVALAI_MODEL = "gemma-3n-e4b-it"

# ----------------------
# Enhanced Core Functions
# ----------------------
def load_dataset_from_bytes(b: bytes, source_hint: str = "") -> Dict[str, str]:
    try:
        data = json.loads(b.decode("utf-8"))
    except Exception as e:
        raise ValueError(f"Could not read JSON from {source_hint}: {e}")
    if isinstance(data, dict) and "dataset" in data:
        records = data["dataset"]
    elif isinstance(data, list):
        records = data
    else:
        raise ValueError(f"Unrecognized JSON structure in {source_hint}.")
    mapping = {}
    for rec in records:
        img = rec.get("source_image") or rec.get("image_id") or rec.get("image") or rec.get("id", f"item_{len(mapping)+1:05d}")
        turtle = rec.get("rdf_graph_turtle") or rec.get("turtle") or rec.get("triples_turtle", "")
        if not isinstance(turtle, str):
            turtle = json.dumps(turtle, ensure_ascii=False)
        mapping[str(img)] = turtle
    return mapping

ALLOWED_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

def index_images(images_dir: Optional[Path]) -> Dict[str, str]:
    idx = {}
    if not images_dir or not images_dir.exists():
        return idx
    for p in images_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in ALLOWED_EXTS:
            idx.setdefault(p.stem, str(p.resolve()))
    return idx

def find_image_path_indexed(idx: Dict[str, str], name: str) -> Optional[str]:
    stem = Path(name).stem
    return idx.get(stem, None)

def _truncate_smart(s: str, max_chars: int = MAX_CHARS) -> str:
    """Smart truncation that tries to preserve structure"""
    s = s or ""
    if len(s) <= max_chars:
        return s
    
    # Try to truncate at natural boundaries
    truncated = s[:max_chars]
    
    # Look for last complete line or triple
    last_newline = truncated.rfind('\n')
    last_period = truncated.rfind('.')
    
    if last_newline > max_chars * 0.8:  # If newline is in last 20%
        truncated = s[:last_newline]
    elif last_period > max_chars * 0.8:  # If period is in last 20%
        truncated = s[:last_period + 1]
    
    return truncated + "\n... [TRUNCATED]"

def _calculate_backoff_delay(attempt: int, base_delay: float = EXPONENTIAL_BACKOFF_BASE) -> float:
    """Calculate exponential backoff delay with jitter"""
    delay = min(MAX_BACKOFF_DELAY, base_delay ** attempt)
    # Add jitter (±25%)
    jitter = delay * 0.25 * (2 * random.random() - 1)
    return max(1.0, delay + jitter)

def _sleep_with_backoff(attempt: int, retry_after: Optional[str] = None, rate_limit_error: bool = False):
    """Enhanced backoff with different strategies for different error types"""
    if retry_after:
        try:
            sleep_time = float(retry_after)
            print(f"⏱️ API requested {sleep_time}s delay")
            time.sleep(min(sleep_time, MAX_BACKOFF_DELAY))
            return
        except (ValueError, TypeError):
            pass
    
    if rate_limit_error:
        # More aggressive backoff for rate limit errors
        delay = _calculate_backoff_delay(attempt, base_delay=5.0)
        print(f"🚫 Rate limit backoff: {delay:.1f}s")
    else:
        # Standard backoff for other errors
        delay = _calculate_backoff_delay(attempt)
        print(f"⏳ Retry backoff: {delay:.1f}s")
    
    time.sleep(delay)

def _extract_json_from_text(txt: str) -> dict:
    """Enhanced JSON extraction with better error handling"""
    if not txt or not txt.strip():
        raise ValueError("Empty or whitespace-only response text.")
    
    txt = txt.strip()
    
    # Handle code fences
    if "```json" in txt:
        start = txt.find("```json") + 7
        end = txt.find("```", start)
        if end != -1:
            txt = txt[start:end].strip()
    elif "```" in txt:
        # Generic code fence
        start = txt.find("```")
        end = txt.rfind("```")
        if start != end and start != -1:
            txt = txt[start+3:end].strip()
    
    # Try direct parse first
    try:
        result = json.loads(txt)
        if isinstance(result, dict):
            return result
    except json.JSONDecodeError:
        pass
    
    # Try to find JSON object bounds
    brace_start = txt.find("{")
    if brace_start == -1:
        raise ValueError(f"No opening brace found in response: {txt[:200]}")
    
    # Find matching closing brace
    brace_count = 0
    for i, char in enumerate(txt[brace_start:], brace_start):
        if char == "{":
            brace_count += 1
        elif char == "}":
            brace_count -= 1
            if brace_count == 0:
                json_str = txt[brace_start:i+1]
                try:
                    return json.loads(json_str)
                except json.JSONDecodeError:
                    break
    
    raise ValueError(f"Could not extract valid JSON from response: {txt[:300]}")

def _validate_and_fix_response(response_dict: dict) -> Tuple[str, List[str], List[str], List[str]]:
    """Validate and fix the API response structure"""
    winner = response_dict.get("winner", "Tie")
    if winner not in {"A", "B", "Tie"}:
        print(f"⚠️ Invalid winner '{winner}', defaulting to 'Tie'")
        winner = "Tie"
    
    reasons = response_dict.get("reasons", [])
    if not isinstance(reasons, list):
        reasons = [str(reasons)] if reasons else ["No reasons provided"]
    
    errA = response_dict.get("major_errors_A", [])
    if not isinstance(errA, list):
        errA = [str(errA)] if errA else []
    
    errB = response_dict.get("major_errors_B", [])
    if not isinstance(errB, list):
        errB = [str(errB)] if errB else []
    
    return winner, reasons, errA, errB

def avalai_judge_robust(triples_A: str, label_A: str,
                       triples_B: str, label_B: str,
                       api_key: str, base_url: str, model_name: str,
                       rate_limiter: RateLimiter,
                       debug_prefix: str = "") -> Tuple[str, List[str], List[str], List[str]]:
    """
    Enhanced AvalAI judge with robust error handling and rate limiting
    """
    if not api_key:
        raise ValueError("AVALAI_API_KEY is not set. Configure Kaggle Secret 'avalai_api' or env var 'AVALAI_API_KEY'.")

    endpoint = f"{base_url}/chat/completions"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}",
        "Accept": "application/json",
        "User-Agent": "PairwiseComparison/1.0"
    }

    # Enhanced instruction with clearer formatting requirements
    instruction = (
        "You are evaluating RDF triples extracted from the same image by two different methods. "
        "Analyze both extractions for correctness, completeness, and RDF validity.\n\n"
        "IMPORTANT: Respond with ONLY a valid JSON object in this exact format:\n"
        "{\n"
        '  "winner": "A",\n'
        '  "reasons": ["reason 1", "reason 2"],\n'
        '  "major_errors_A": ["error 1 in method A"],\n'
        '  "major_errors_B": ["error 1 in method B"]\n'
        "}\n\n"
        "Rules:\n"
        "- winner must be exactly 'A', 'B', or 'Tie'\n"
        "- All arrays must contain strings\n"
        "- Do not include extra text before or after the JSON\n"
        "- Consider: correctness relative to image content, completeness, RDF syntax validity\n"
        "- Longer extractions are not automatically better"
    )

    # Smart truncation
    triples_A_tr = _truncate_smart(triples_A, MAX_CHARS)
    triples_B_tr = _truncate_smart(triples_B, MAX_CHARS)

    user_prompt = (
        f"{instruction}\n\n"
        f"Method A ({label_A}) triples:\n{triples_A_tr}\n\n"
        f"Method B ({label_B}) triples:\n{triples_B_tr}\n"
    )

    # Simplified payload - remove potentially problematic parameters
    payload = {
        "model": model_name,
        "messages": [
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.1,  # Slightly increased for more natural responses
        "max_tokens": 800,   # Reduced to avoid long responses
    }

    if PRINT_REQUEST_SIZES:
        print(f"{debug_prefix} [sizes] A:{len(triples_A)}->{len(triples_A_tr)} chars | "
              f"B:{len(triples_B)}->{len(triples_B_tr)} chars | prompt:{len(user_prompt)} chars")

    last_error = None
    
    for attempt in range(MAX_RETRIES):
        try:
            # Wait according to rate limiter
            rate_limiter.wait_if_needed()
            
            # Make request
            response = requests.post(
                endpoint, 
                headers=headers, 
                json=payload, 
                timeout=REQUEST_TIMEOUT
            )
            
            status_code = response.status_code
            
            if PRINT_RESPONSE_SNIPPET:
                snippet = response.text[:400].replace("\n", " ")
                print(f"{debug_prefix} [HTTP {status_code}] response: {snippet}")

            # Handle different status codes
            if status_code == 200:
                # Success case
                try:
                    data = response.json()
                    content_text = (
                        data.get("choices", [{}])[0]
                            .get("message", {})
                            .get("content", "")
                    ).strip()
                    
                    if not content_text:
                        raise ValueError("Empty response content")
                    
                    parsed_response = _extract_json_from_text(content_text)
                    winner, reasons, errA, errB = _validate_and_fix_response(parsed_response)
                    
                    rate_limiter.record_success()
                    return winner, reasons, errA, errB
                    
                except (json.JSONDecodeError, ValueError, KeyError) as parse_error:
                    print(f"{debug_prefix} ⚠️ Parse error: {parse_error}")
                    last_error = parse_error
                    if attempt < MAX_RETRIES - 1:
                        time.sleep(1)  # Brief pause before retry
                        continue
                    
            elif status_code == 429:
                # Rate limit error
                rate_limiter.record_error()
                print(f"{debug_prefix} 🚫 Rate limit exceeded (attempt {attempt + 1}/{MAX_RETRIES})")
                
                if attempt < MAX_RETRIES - 1:
                    _sleep_with_backoff(
                        attempt, 
                        response.headers.get("Retry-After"),
                        rate_limit_error=True
                    )
                    continue
                
            elif status_code in [400, 401, 403]:
                # Client errors - don't retry
                try:
                    error_data = response.json()
                    error_msg = error_data.get("error", {}).get("message", "Unknown error")
                except:
                    error_msg = response.text[:500]
                
                print(f"{debug_prefix} ❌ Client error {status_code}: {error_msg}")
                raise requests.exceptions.HTTPError(f"Client error {status_code}: {error_msg}")
                
            elif status_code in [500, 502, 503, 504]:
                # Server errors - retry with backoff
                rate_limiter.record_error()
                print(f"{debug_prefix} 🔄 Server error {status_code} (attempt {attempt + 1}/{MAX_RETRIES})")
                
                if attempt < MAX_RETRIES - 1:
                    _sleep_with_backoff(attempt)
                    continue
                    
            else:
                # Other errors
                print(f"{debug_prefix} ❓ Unexpected status {status_code}")
                response.raise_for_status()

        except requests.exceptions.Timeout:
            last_error = "Request timeout"
            print(f"{debug_prefix} ⏰ Timeout (attempt {attempt + 1}/{MAX_RETRIES})")
            if attempt < MAX_RETRIES - 1:
                _sleep_with_backoff(attempt)
                continue
                
        except requests.exceptions.ConnectionError as conn_error:
            last_error = conn_error
            print(f"{debug_prefix} 🔌 Connection error (attempt {attempt + 1}/{MAX_RETRIES})")
            if attempt < MAX_RETRIES - 1:
                _sleep_with_backoff(attempt)
                continue
                
        except Exception as unexpected_error:
            last_error = unexpected_error
            print(f"{debug_prefix} ❌ Unexpected error: {unexpected_error}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(2)
                continue
    
    # All retries exhausted
    rate_limiter.record_error()
    error_msg = f"Failed after {MAX_RETRIES} attempts. Last error: {last_error}"
    print(f"{debug_prefix} 💥 {error_msg}")
    
    # Return neutral result instead of crashing
    return "Tie", [f"API_ERROR: {error_msg}"], [], []

# ----------------------
# Main Workflow
# ----------------------
def main():
    random.seed(RANDOM_SEED)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # Initialize rate limiter
    rate_limiter = RateLimiter(MIN_SLEEP_BETWEEN_ITEMS, MAX_SLEEP_BETWEEN_ITEMS)
    
    # Error logging
    ERROR_LOG = OUTPUT_DIR / "avalai_errors.log"
    
    def log_error(msg: str):
        timestamp = datetime.now().isoformat()
        try:
            with ERROR_LOG.open("a", encoding="utf-8") as f:
                f.write(f"[{timestamp}] {msg}\n")
        except Exception:
            pass

    print("📂 Loading datasets...")

    if not os.path.exists(ONE_SHOT_JSON_PATH) or not os.path.exists(FEW_SHOT_JSON_PATH):
        raise FileNotFoundError("One or both JSON file paths are incorrect. Please check the paths.")

    one_name = Path(ONE_SHOT_JSON_PATH).name
    few_name = Path(FEW_SHOT_JSON_PATH).name

    uploaded = {
        one_name: Path(ONE_SHOT_JSON_PATH).read_bytes(),
        few_name: Path(FEW_SHOT_JSON_PATH).read_bytes()
    }
    print(f"📄 One-Shot file: {one_name}")
    print(f"📄 Few-Shot file: {few_name}")

    one_map = load_dataset_from_bytes(uploaded[one_name], source_hint=one_name)
    few_map = load_dataset_from_bytes(uploaded[few_name], source_hint=few_name)

    shared = sorted(set(one_map.keys()) & set(few_map.keys()))
    print(f"✅ Shared items: {len(shared)}")

    # Index images
    IMAGES_DIR = Path(IMAGES_DIR_PATH)
    if IMAGES_DIR.exists():
        print(f"🗂️ Indexing images in folder: {IMAGES_DIR}")
        IMAGE_INDEX = index_images(IMAGES_DIR)
        print(f"  -> Found {len(IMAGE_INDEX)} images")
    else:
        print(f"ℹ️ Images folder not found at: {IMAGES_DIR}")
        IMAGE_INDEX = {}

    # Prepare output files
    summary_csv = OUTPUT_DIR / "pairwise_summary.csv"
    pairs_jsonl = OUTPUT_DIR / "pairwise_pairs.jsonl"
    prompt_txt = OUTPUT_DIR / "judge_prompt.txt"
    judge_csv = OUTPUT_DIR / "judge_report.csv"
    aggregate_csv = OUTPUT_DIR / "judge_aggregate.csv"

    # Save prompt template
    prompt_txt.write_text(
    """Enhanced Pairwise RDF evaluation Prompt:

    You are evaluating RDF triples extracted from the same image by two methods.
    Analyze both extractions for correctness, completeness, and RDF validity.

    Respond with ONLY a valid JSON object in this exact format:
    {
      "winner": "A",
      "reasons": ["reason 1", "reason 2"],
      "major_errors_A": ["error 1 in method A"],
      "major_errors_B": ["error 1 in method B"]
    }

    Rules:
    - winner must be exactly 'A', 'B', or 'Tie'
    - All arrays must contain strings
    - No extra text before or after JSON
    - Consider: correctness, completeness, RDF syntax validity
    - Longer extractions are not automatically better
    """, encoding="utf-8"
    )

    # Build comparison pairs
    rows = []
    with pairs_jsonl.open("w", encoding="utf-8") as jout:
        for img in shared:
            turtle_one = one_map.get(img, "")
            turtle_few = few_map.get(img, "")
            
            # Randomize A/B assignment
            if random.random() < 0.5:
                A_label, B_label = "One-Shot", "Few-Shot"
                A_turtle, B_turtle = turtle_one, turtle_few
            else:
                A_label, B_label = "Few-Shot", "One-Shot"
                A_turtle, B_turtle = turtle_few, turtle_one

            img_path = find_image_path_indexed(IMAGE_INDEX, img)

            rows.append({
                "image_id": img, 
                "img_path": img_path or "", 
                "A_model": A_label, 
                "B_model": B_label
            })
            
            jout.write(json.dumps({
                "image_id": img,
                "image_path": img_path,
                "A_label": A_label,
                "B_label": B_label,
                "triples_A_turtle": A_turtle,
                "triples_B_turtle": B_turtle,
                "seed": RANDOM_SEED
            }, ensure_ascii=False) + "\n")

    # Write summary
    if rows:
        with summary_csv.open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)

    print(f"\n📝 Generated {len(rows)} comparison pairs")
    print(f"📝 Files: {summary_csv}, {pairs_jsonl}, {prompt_txt}")

    # ----------------------
    # Run AvalAI judging
    # ----------------------
    if JUDGE_MODE == "avalai":
        print(f"\n🤖 Starting robust judge: {JUDGE_MODE} with model {AVALAI_MODEL}")
        print(f"🛡️ Rate limiting: {MIN_SLEEP_BETWEEN_ITEMS}-{MAX_SLEEP_BETWEEN_ITEMS}s between requests")
        
        # Load pairs
        with pairs_jsonl.open("r", encoding="utf-8") as f:
            lines = [json.loads(ln) for ln in f]

        results = []
        total_items = len(lines)
        errors_count = 0
        start_time = time.time()

        print(f"🚀 Processing {total_items} items...")
        
        for i, rec in enumerate(lines, start=1):
            prefix = f"[{i}/{total_items}]"
            item_start_time = time.time()
            
            print(f"\n{prefix} Processing: {rec['image_id']}")
            
            try:
                winner, reasons, errA, errB = avalai_judge_robust(
                    rec["triples_A_turtle"], rec["A_label"],
                    rec["triples_B_turtle"], rec["B_label"],
                    api_key=AVALAI_API_KEY, 
                    base_url=BASE_URL, 
                    model_name=AVALAI_MODEL,
                    rate_limiter=rate_limiter,
                    debug_prefix=prefix
                )
                
                item_time = time.time() - item_start_time
                
                if "API_ERROR" in str(reasons):
                    errors_count += 1
                    status_emoji = "⚠️"
                else:
                    status_emoji = "✅"
                
                print(f"{prefix} {status_emoji} Winner: {winner} | "
                      f"Time: {item_time:.1f}s | "
                      f"Reasons: {'; '.join(reasons)[:150]}")
                
                # Log detailed info
                log_error(f"Item {i}: {rec['image_id']} -> Winner: {winner}, "
                         f"Reasons: {reasons}, ErrorsA: {errA}, ErrorsB: {errB}")
                
            except Exception as e:
                errors_count += 1
                winner, reasons, errA, errB = "Tie", [f"FATAL_ERROR: {str(e)[:200]}"], [], []
                print(f"{prefix} ❌ Fatal error: {e}")
                log_error(f"FATAL ERROR Item {i} ({rec['image_id']}): {e}")

            results.append({
                "image_id": rec["image_id"],
                "A_label": rec["A_label"],
                "B_label": rec["B_label"],
                "winner": winner,
                "reasons": " | ".join(reasons),
                "errors_A": " | ".join(errA),
                "errors_B": " | ".join(errB)
            })

            # Progress update
            if i % 10 == 0:
                elapsed = time.time() - start_time
                rate = i / elapsed if elapsed > 0 else 0
                eta = (total_items - i) / rate if rate > 0 else 0
                print(f"📊 Progress: {i}/{total_items} ({i/total_items*100:.1f}%) | "
                      f"Rate: {rate:.2f} items/min | ETA: {eta/60:.1f}min | Errors: {errors_count}")

        # Final statistics
        total_time = time.time() - start_time
        print(f"\n🏁 Completed processing in {total_time/60:.1f} minutes")
        print(f"📈 Average rate: {len(results) / total_time * 60:.2f} items/minute")
        print(f"❌ Total errors: {errors_count}/{len(results)} ({errors_count/len(results)*100:.1f}%)")

        # Calculate aggregates
        winA = sum(1 for r in results if r["winner"] == "A")
        winB = sum(1 for r in results if r["winner"] == "B")
        ties = sum(1 for r in results if r["winner"] == "Tie")
        total = len(results)

        # Save detailed results
        if results:
            with judge_csv.open("w", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
                writer.writeheader()
                writer.writerows(results)

            with aggregate_csv.open("w", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=[
                    "total", "A_wins", "B_wins", "ties", "errors", 
                    "processing_time_min", "avg_rate_per_min"
                ])
                writer.writeheader()
                writer.writerow({
                    "total": total, 
                    "A_wins": winA, 
                    "B_wins": winB, 
                    "ties": ties,
                    "errors": errors_count,
                    "processing_time_min": round(total_time / 60, 2),
                    "avg_rate_per_min": round(len(results) / total_time * 60, 2)
                })

        print(f"\n🧮 Final Results:")
        print(f"   Total: {total}")
        print(f"   A wins: {winA} ({winA/total*100:.1f}%)")
        print(f"   B wins: {winB} ({winB/total*100:.1f}%)")
        print(f"   Ties: {ties} ({ties/total*100:.1f}%)")
        print(f"   Errors: {errors_count} ({errors_count/total*100:.1f}%)")
        print(f"\n📝 Output files:")
        print(f"   - {judge_csv}")
        print(f"   - {aggregate_csv}")
        print(f"   - {ERROR_LOG}")

    # Download files in Colab
    if IN_COLAB:
        print("\n⬇️ Downloading results...")
        for filepath in [summary_csv, pairs_jsonl, prompt_txt, judge_csv, aggregate_csv, ERROR_LOG]:
            if filepath.exists():
                try:
                    files.download(str(filepath))
                    print(f"✅ Downloaded: {filepath.name}")
                except Exception as e:
                    print(f"❌ Failed to download {filepath.name}: {e}")

    print("\n🎉 All done!")


# ----------------------
# Execute main function
# ----------------------
if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n⏹️ Process interrupted by user")
        sys.exit(1)
    except Exception as e:
        print(f"\n💥 Fatal error in main execution: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)

In [ ]:
print(f"\n📁 All outputs saved to: {OUTPUT_DIR.resolve()}")
for p in [summary_csv, pairs_jsonl, prompt_txt, judge_csv, aggregate_csv]:
    if p.exists():
        print(" -", p.name)
